# BIRD Text-to-SQL data viewer

交互查看 Stage A/B SFT 与 RLVR JSONL。viewer 使用只读 `mmap`，不会把 91 万条样本整体载入内存。

- `Random`：未建立索引时按文件字节位置抽样（更容易看到长样本）；建立索引后按行均匀抽样。
- `Build exact row index`：可选。扫描一次当前文件，并用紧凑的 64-bit offset 数组支持精确行号跳转。
- Schema、system prompt 和 raw JSON 默认折叠，避免长样本撑满页面。

In [ ]:
from __future__ import annotations

from array import array
from pathlib import Path
import bisect
import html
import json
import mmap
import random
import re

from IPython.display import HTML, display
import ipywidgets as widgets

DATA_ROOT = Path("/data/ximo/bird-text2sql-rl/data")
DATASETS = {
    "SFT wide / train": DATA_ROOT / "sft-v2-wide/train.jsonl",
    "SFT wide / validation": DATA_ROOT / "sft-v2-wide/validation.jsonl",
    "SFT BIRD / train": DATA_ROOT / "sft-v2-bird/train.jsonl",
    "SFT BIRD / validation": DATA_ROOT / "sft-v2-bird/validation.jsonl",
    "RLVR / train": DATA_ROOT / "rlvr-v2/train.jsonl",
    "RLVR / validation": DATA_ROOT / "rlvr-v2/validation.jsonl",
    "RLVR / rejections": DATA_ROOT / "rlvr-v2/rejections.jsonl",
}
DATASETS = {name: path for name, path in DATASETS.items() if path.is_file()}
if not DATASETS:
    raise FileNotFoundError(f"No JSONL datasets found below {DATA_ROOT}")

print(f"Found {len(DATASETS)} JSONL files under {DATA_ROOT}")

Found 7 JSONL files under /data/ximo/bird-text2sql-rl/data


In [ ]:
def load_manifest(path: Path) -> dict:
    manifest_path = path.parent / "manifest.json"
    return json.loads(manifest_path.read_text()) if manifest_path.is_file() else {}


rows = []
for name, path in DATASETS.items():
    manifest = load_manifest(path)
    split = "rejected" if path.stem == "rejections" else path.stem
    count = manifest.get("counts", {}).get(split, "—")
    rows.append(
        f"<tr><td>{html.escape(name)}</td><td>{count:,}</td>"
        f"<td>{path.stat().st_size / 2**30:.2f} GiB</td>"
        f"<td><code>{html.escape(str(path))}</code></td></tr>"
        if isinstance(count, int)
        else f"<tr><td>{html.escape(name)}</td><td>{count}</td>"
        f"<td>{path.stat().st_size / 2**30:.2f} GiB</td>"
        f"<td><code>{html.escape(str(path))}</code></td></tr>"
    )
display(HTML(
    "<table><thead><tr><th>Dataset</th><th>Manifest rows</th><th>Size</th><th>Path</th></tr></thead>"
    f"<tbody>{''.join(rows)}</tbody></table>"
))

Dataset,Manifest rows,Size,Path
SFT wide / train,"914,320",7.00 GiB,/data/ximo/bird-text2sql-rl/data/sft-v2-wide/train.jsonl
SFT wide / validation,"1,836",0.01 GiB,/data/ximo/bird-text2sql-rl/data/sft-v2-wide/validation.jsonl
SFT BIRD / train,"2,064",0.03 GiB,/data/ximo/bird-text2sql-rl/data/sft-v2-bird/train.jsonl
SFT BIRD / validation,398,0.01 GiB,/data/ximo/bird-text2sql-rl/data/sft-v2-bird/validation.jsonl
RLVR / train,"2,050",0.02 GiB,/data/ximo/bird-text2sql-rl/data/rlvr-v2/train.jsonl
RLVR / validation,396,0.01 GiB,/data/ximo/bird-text2sql-rl/data/rlvr-v2/validation.jsonl
RLVR / rejections,16,0.00 GiB,/data/ximo/bird-text2sql-rl/data/rlvr-v2/rejections.jsonl


In [ ]:
class JsonlReader:
    """Read one JSONL record at a time without loading the file."""

    def __init__(self, path: Path):
        self.path = path
        self.file = path.open("rb")
        self.data = mmap.mmap(self.file.fileno(), 0, access=mmap.ACCESS_READ)
        self.size = len(self.data)
        self.current_start = 0
        self.offsets: array | None = None

    def close(self):
        self.data.close()
        self.file.close()

    def containing_start(self, byte_position: int) -> int:
        if not self.size:
            raise ValueError(f"Empty JSONL file: {self.path}")
        position = min(max(0, byte_position), self.size - 1)
        delimiter = self.data.rfind(b"\n", 0, position)
        return delimiter + 1 if delimiter >= 0 else 0

    def read_at(self, start: int) -> dict:
        start = self.containing_start(start)
        end = self.data.find(b"\n", start)
        end = self.size if end < 0 else end
        self.current_start = start
        return json.loads(self.data[start:end])

    def first(self) -> dict:
        return self.read_at(0)

    def next(self) -> dict:
        end = self.data.find(b"\n", self.current_start)
        if end < 0 or end + 1 >= self.size:
            return self.read_at(self.current_start)
        return self.read_at(end + 1)

    def previous(self) -> dict:
        if self.current_start == 0:
            return self.read_at(0)
        delimiter = self.data.rfind(b"\n", 0, self.current_start - 1)
        return self.read_at(delimiter + 1 if delimiter >= 0 else 0)

    def at_percent(self, percent: float) -> dict:
        return self.read_at(int((self.size - 1) * min(max(percent, 0.0), 100.0) / 100.0))

    def random(self) -> dict:
        if self.offsets is not None:
            return self.read_at(self.offsets[random.randrange(len(self.offsets))])
        return self.read_at(random.randrange(self.size))

    def build_index(self) -> int:
        offsets = array("Q", [0]) if self.size else array("Q")
        position = 0
        while position < self.size:
            delimiter = self.data.find(b"\n", position)
            if delimiter < 0:
                break
            position = delimiter + 1
            if position < self.size:
                offsets.append(position)
        self.offsets = offsets
        return len(offsets)

    def at_row(self, row: int) -> dict:
        if self.offsets is None:
            raise RuntimeError("Build the exact row index first")
        return self.read_at(self.offsets[row])

    @property
    def current_row(self) -> int | None:
        if self.offsets is None:
            return None
        return bisect.bisect_left(self.offsets, self.current_start)


TAGS = ("requirements", "reasoning", "verification", "sql")


def extract_tag(text: str, tag: str) -> str:
    match = re.search(rf"<{tag}>(.*?)</{tag}>", text, flags=re.DOTALL | re.IGNORECASE)
    return match.group(1).strip() if match else ""


def prompt_parts(user: str) -> dict[str, str]:
    def capture(pattern: str) -> str:
        match = re.search(pattern, user, flags=re.DOTALL | re.IGNORECASE)
        return match.group(1).strip() if match else ""

    wide_schema = capture(r"Database Schema:\s*(.*?)(?:\nThis schema describes|\n\nQuestion:)")
    bird_schema = capture(r"Schema.*?:\s*(.*?)\n\nExternal evidence:")
    return {
        "database": capture(r"(?:^|\n)Database:\s*(.*?)\n"),
        "schema": wide_schema or bird_schema,
        "evidence": capture(r"External evidence:\s*(.*?)\n\nQuestion:"),
        "question": capture(r"Question:\s*(.*?)(?:\n\nInstructions:|$)"),
    }


def section(title: str, value: object, *, collapsed: bool = False, css_class: str = "") -> str:
    text = "" if value is None else str(value)
    body = f'<pre class="{css_class}">{html.escape(text)}</pre>'
    if collapsed:
        return f"<details><summary>{html.escape(title)}</summary>{body}</details>"
    return f"<section><h3>{html.escape(title)}</h3>{body}</section>"


def render_record(record: dict) -> HTML:
    messages = record.get("messages") or []
    by_role = {item.get("role"): item.get("content", "") for item in messages}
    user = by_role.get("user", "")
    assistant = by_role.get("assistant", "")
    parts = prompt_parts(user) if user else {}

    metadata = {key: value for key, value in record.items() if key not in {"messages", "gold_rows_json", "SQL"}}
    cards = [section("Metadata", json.dumps(metadata, ensure_ascii=False, indent=2), css_class="meta")]
    question = parts.get("question") or record.get("question", "")
    evidence = parts.get("evidence") or record.get("evidence", "")
    if question:
        cards.append(section("Question", question))
    if evidence:
        cards.append(section("External evidence", evidence, css_class="evidence"))

    if assistant:
        cards.extend([
            section("Requirements", extract_tag(assistant, "requirements") or "(empty)"),
            section("Reasoning", extract_tag(assistant, "reasoning")),
            section("Verification", extract_tag(assistant, "verification") or "(empty)"),
            section("SQL", extract_tag(assistant, "sql"), css_class="sql"),
        ])
    elif record.get("SQL") is not None:
        cards.append(section("Gold SQL", record.get("SQL"), css_class="sql"))
        cards.append(section("Gold rows", record.get("gold_rows_json", ""), collapsed=True))

    schema = parts.get("schema", "")
    if schema:
        cards.append(section("Schema", schema, collapsed=True))
    if by_role.get("system"):
        cards.append(section("System prompt", by_role["system"], collapsed=True))
    if user:
        cards.append(section("Raw user prompt", user, collapsed=True))
    cards.append(section("Raw JSON", json.dumps(record, ensure_ascii=False, indent=2), collapsed=True))

    style = """
    <style>
      .bird-viewer {font-family: ui-sans-serif, system-ui; max-width: 1200px}
      .bird-viewer section, .bird-viewer details {border:1px solid #d0d7de; border-radius:8px; padding:10px 14px; margin:10px 0}
      .bird-viewer h3 {font-size:15px; margin:0 0 8px}
      .bird-viewer summary {font-weight:600; cursor:pointer}
      .bird-viewer pre {white-space:pre-wrap; overflow-wrap:anywhere; margin:0; font-size:13px; line-height:1.45}
      .bird-viewer .sql {background:#f6f8fa; color:#0550ae}
      .bird-viewer .evidence {background:#fff8c5}
      .bird-viewer .meta {color:#57606a}
    </style>
    """
    return HTML(style + f"<div class='bird-viewer'>{''.join(cards)}</div>")

In [ ]:
dataset_select = widgets.Dropdown(options=list(DATASETS), description="Dataset:", layout=widgets.Layout(width="520px"))
previous_button = widgets.Button(description="← Previous")
next_button = widgets.Button(description="Next →")
random_button = widgets.Button(description="Random", button_style="info")
percent_input = widgets.BoundedFloatText(value=0.0, min=0.0, max=100.0, step=0.1, description="File %:")
percent_button = widgets.Button(description="Jump")
index_button = widgets.Button(description="Build exact row index", button_style="warning")
row_input = widgets.BoundedIntText(value=0, min=0, max=0, description="Row:", disabled=True)
row_button = widgets.Button(description="Go to row", disabled=True)
status = widgets.HTML()
output = widgets.Output()
state = {"reader": None}


def show(record: dict):
    reader = state["reader"]
    row = reader.current_row
    row_text = f"row {row:,} / {len(reader.offsets) - 1:,}" if row is not None else "row index not built"
    percent = 100.0 * reader.current_start / max(reader.size, 1)
    status.value = (
        f"<b>{html.escape(dataset_select.value)}</b> · {row_text} · "
        f"byte {reader.current_start:,} / {reader.size:,} ({percent:.3f}%)"
    )
    percent_input.value = min(100.0, percent)
    if row is not None:
        row_input.value = row
    with output:
        output.clear_output(wait=True)
        display(render_record(record))


def open_dataset(change=None):
    old_reader = state.get("reader")
    if old_reader is not None:
        old_reader.close()
    state["reader"] = JsonlReader(DATASETS[dataset_select.value])
    row_input.disabled = True
    row_button.disabled = True
    row_input.max = 0
    index_button.disabled = False
    index_button.description = "Build exact row index"
    show(state["reader"].first())


def build_index(_):
    reader = state["reader"]
    index_button.disabled = True
    index_button.description = "Scanning…"
    status.value = "<b>Scanning JSONL once to build the in-memory row index…</b>"
    try:
        count = reader.build_index()
        row_input.max = max(0, count - 1)
        row_input.disabled = False
        row_button.disabled = False
        index_button.description = f"Indexed {count:,} rows"
        show(reader.read_at(reader.current_start))
    except Exception:
        index_button.disabled = False
        index_button.description = "Build exact row index"
        raise


dataset_select.observe(open_dataset, names="value")
previous_button.on_click(lambda _: show(state["reader"].previous()))
next_button.on_click(lambda _: show(state["reader"].next()))
random_button.on_click(lambda _: show(state["reader"].random()))
percent_button.on_click(lambda _: show(state["reader"].at_percent(percent_input.value)))
index_button.on_click(build_index)
row_button.on_click(lambda _: show(state["reader"].at_row(row_input.value)))

open_dataset()
display(widgets.VBox([
    dataset_select,
    widgets.HBox([previous_button, next_button, random_button]),
    widgets.HBox([percent_input, percent_button]),
    widgets.HBox([index_button, row_input, row_button]),
    status,
    output,
]))

## 使用提示

在仓库根目录启动：

```bash
jupyter lab notebooks/data_viewer.ipynb
```

如果修改了数据目录，只需调整第二个 cell 的 `DATA_ROOT`。精确索引只保存在当前 kernel 内存中，不会在数据目录生成额外文件。